# 🚀 Fine-tune Qwen 2.5 0.5B for Autonomous Jobstreet Agent
Trains on on-device screening questions & tool-calling navigation trajectories.
Uses **Unsloth (2x faster, 70% less VRAM)** on a free Google Colab T4 GPU.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

### 1. Load Qwen2.5-0.5B-Instruct in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 2. Configure LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Load & Format Dataset (`fine_tuning_dataset.jsonl`)
Upload `fine_tuning_dataset.jsonl` directly to Colab files panel before running this.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="fine_tuning_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print("Sample formatted prompt:\n", dataset[0]["text"][:300])

### 4. Train the Model (Takes ~3-5 mins on T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        warmup_steps = 10,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

### 5. Export to GGUF (Ready for Termux / llama.cpp on Phone!)

In [ ]:
# Save quantized GGUF format (Q8_0 for perfect accuracy or Q4_K_M for smaller file size)
model.save_pretrained_gguf("qwen2.5_0.5b_jobaut", tokenizer, quantization_method = "q8_0")
print("GGUF Export Complete! Download the .gguf file from the file explorer.")